# Coding a Corpus, and Cohen's Kappa

**Computational Text Analysis Pack · Module 17 — Annotation and Agreement**

*Technique:* Coding schemes; raw agreement vs chance-corrected agreement

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sagaustus/adh-dh-notebooks/blob/main/06-supervised-classification/17a_coding_and_cohens_kappa.ipynb)

Data: `masakhanews.csv` — from the `data/` folder of this pack.

---

In [ ]:
# Setup — run this first.
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

# The data this notebook needs. It is downloaded from CourseHub automatically;
# uploading by hand is only a fallback if that server is unreachable.
NEEDED = ['masakhanews.csv']
DATA_URL = 'https://coursehub.africandigitalhumanities.ca/ide/pack-data/text/'

def _missing():
    return [f for f in NEEDED if not os.path.exists(f)]

missing = _missing()
if missing:
    # Fetch from CourseHub first, so nobody has to upload anything.
    import urllib.request, urllib.parse
    for _f in list(missing):
        try:
            urllib.request.urlretrieve(DATA_URL + urllib.parse.quote(_f), _f)
            print(f'downloaded {_f}')
        except Exception as _e:
            print(f'could not fetch {_f}: {_e}')
    missing = _missing()

if missing:
    try:
        from google.colab import files
    except ImportError:
        raise SystemExit('Place these next to the notebook: ' + ', '.join(missing))
    # Ask again until everything has arrived. The upload widget returns as soon
    # as you close it, so picking only some of the files would otherwise fail a
    # few lines below with a confusing FileNotFoundError.
    for _ in range(4):
        print('Select ALL of these at once (ctrl-click / cmd-click to multi-select):')
        print('   ' + ', '.join(missing))
        files.upload()
        missing = _missing()
        if not missing:
            break
        print('Still needed: ' + ', '.join(missing))
    if missing:
        raise SystemExit(
            'Missing: ' + ', '.join(missing) + '. Re-run this cell and select '
            'every file listed, or upload them with the folder icon on the left.')

df       = pd.read_csv('masakhanews.csv')
NIGERIAN = ['Hausa', 'Igbo', 'Yoruba', 'Nigerian Pidgin']
# Most of this pack runs on scikit-learn and the standard library. Also
# available: nltk, spacy, gensim and networkx (modules 14-19), Keras
# (case studies), and a GPU pod reached through /ide/api/neural/ for the
# neural modules — never called directly, so no token lives in a notebook.

plt.rcParams['figure.figsize'] = (10, 5.5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25

print(f'Loaded. df has {len(df):,} rows and {df.shape[1]} columns.')

## What this notebook does

Notebook 15d ended on an uncomfortable note: two scholars building the same
category dictionary independently agree only about half the time. That is not a
flaw in dictionaries — it is what happens whenever humans code text, and the
field has a century of practice for measuring it.

The headline number, **raw agreement**, is almost always misleading. Two coders
who both label everything "politics" agree 100% of the time and have demonstrated
nothing. **Cohen's kappa** corrects for exactly that.

## The tooling

`sklearn.metrics.cohen_kappa_score` computes kappa; **NLTK** has
`nltk.metrics.agreement` with several coefficients including Krippendorff's
alpha. Annotation platforms — **INCEpTION**, **brat**, **Label Studio**,
**CATMA** — manage the coding process itself and export the multi-coder files
these statistics consume.

In [ ]:
import re

def tokenise(text):
    """Words, lowercased, Unicode-aware.

    `\\w` matches Unicode letters, so ọ, à, ƙ, ɗ and ɓ survive. It does not
    match combining marks, and a tone mark on ọ or ẹ is always one, so the
    pattern adds the combining-accent range U+0300-U+036F; without it ọ̀rọ̀
    splits into ọ + rọ. The common shortcut `[a-z]+` throws away far more and
    quietly changes every count in this notebook — see notebook 03b.
    """
    return re.findall(r"[\w\u0300-\u036f]+", str(text).lower())

sample = "Àwọn ọmọ ìlú ti ṣe ìpàdé lórí ọ̀rọ̀ ẹ̀kọ́ — ƙungiyar ɗalibai ta yi taro"
print("sample     :", sample)
print("tokens     :", tokenise(sample))
print("bare \\w+   :", re.findall(r"\w+", sample.lower()))
print("naive [a-z]:", re.findall(r"[a-z]+", sample.lower()))
print()
print("The naive pattern splits words at every accented letter; bare \\w+")
print("splits them wherever a tone mark sits on ọ or ẹ.")

In [ ]:
# Two simulated coders applying a scheme to the same documents. Coder A is the
# corpus's own category label; coder B is a second reader who is mostly careful,
# systematically confuses two categories, and occasionally slips.
rng = np.random.default_rng(4)
sample = df.sample(n=400, random_state=1).reset_index(drop=True)
coder_a = sample["category"].tolist()

CONFUSABLE = {"politics": "business", "business": "politics",
              "health": "health", "sports": "sports",
              "entertainment": "sports", "religion": "politics",
              "technology": "business"}
coder_b = []
for label in coder_a:
    r = rng.random()
    if r < 0.72:
        coder_b.append(label)
    elif r < 0.90:
        coder_b.append(CONFUSABLE.get(label, label))
    else:
        coder_b.append(rng.choice(sorted(set(coder_a))))

raw = np.mean([a == b for a, b in zip(coder_a, coder_b)])
print(f"documents coded : {len(coder_a)}")
print(f"raw agreement   : {raw:.3f}")

## Why raw agreement is not enough

Some agreement happens by luck, and how much depends on how many categories
there are and how unevenly they are used. Kappa asks: **how much of the
available room for improvement over chance did the coders actually cover?**

In [ ]:
def cohens_kappa(a, b):
    cats = sorted(set(a) | set(b))
    n = len(a)
    po = sum(x == y for x, y in zip(a, b)) / n
    pa = Counter(a)
    pb = Counter(b)
    pe = sum((pa[c] / n) * (pb[c] / n) for c in cats)
    return (po - pe) / (1 - pe), po, pe

k, po, pe = cohens_kappa(coder_a, coder_b)
print(f"observed agreement (po) : {po:.3f}")
print(f"expected by chance (pe) : {pe:.3f}")
print(f"Cohen's kappa           : {k:.3f}")

from sklearn.metrics import cohen_kappa_score
print(f"sklearn agrees          : {cohen_kappa_score(coder_a, coder_b):.3f}")
print()
print("Landis & Koch's conventional bands: <0.20 slight, 0.21-0.40 fair,")
print("0.41-0.60 moderate, 0.61-0.80 substantial, >0.80 almost perfect.")
print("These are conventions, not thresholds derived from anything.")

In [ ]:
# The trap: kappa depends on how balanced the categories are.
print(f"{'scenario':<34}{'raw':>8}{'kappa':>9}")
print("-" * 51)
for name, keep in [("all seven categories", None),
                   ("only politics + health", {"politics", "health"}),
                   ("only politics + technology", {"politics", "technology"})]:
    if keep:
        idx = [i for i, c in enumerate(coder_a) if c in keep and coder_b[i] in keep]
        a2 = [coder_a[i] for i in idx]
        b2 = [coder_b[i] for i in idx]
    else:
        a2, b2 = coder_a, coder_b
    if len(set(a2)) > 1:
        kk, pp, _ = cohens_kappa(a2, b2)
        print(f"{name:<34}{pp:>8.3f}{kk:>9.3f}  (n={len(a2)})")
print()
print("Same coders, same care, very different kappa. A kappa reported without")
print("the category distribution behind it is not interpretable.")

## Where the disagreement is

A single number tells you there is a problem. A confusion matrix tells you where,
and that is what you take back into a revised coding manual.

In [ ]:
cats = sorted(set(coder_a))
mat = pd.DataFrame(0, index=cats, columns=cats)
for a, b in zip(coder_a, coder_b):
    mat.loc[a, b] += 1
print("rows = coder A, columns = coder B")
print(mat.to_string())
print()
off = [(a, b, mat.loc[a, b]) for a in cats for b in cats if a != b]
off.sort(key=lambda t: -t[2])
print("Biggest disagreements:")
for a, b, n in off[:5]:
    print(f"   {a} coded as {b}: {n}")
print()
print("A systematic confusion is a scheme problem, not a coder problem. The fix")
print("is a clearer boundary in the manual, then re-code.")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(mat.values, cmap="Blues")
ax.set_xticks(range(len(cats)))
ax.set_xticklabels(cats, rotation=45, ha="right", fontsize=8)
ax.set_yticks(range(len(cats)))
ax.set_yticklabels(cats, fontsize=8)
ax.set_xlabel("coder B")
ax.set_ylabel("coder A")
ax.set_title(f"Agreement matrix (kappa = {k:.3f})")
fig.colorbar(im, ax=ax, shrink=.8)
plt.tight_layout()

### 🔧 Your turn 1

Change coder B's careful rate from 0.72 to 0.95 and re-run. How much does kappa move compared with raw agreement?

<details markdown="1">
<summary><b>What you should have seen</b> — click to expand</summary>

Kappa moves further than raw agreement does, because it is measuring the
distance travelled above chance rather than the absolute hit rate. That
sensitivity is the reason to prefer it — but it is also why kappa is unstable on
small samples and skewed category distributions, which is the standard criticism
of it and the reason Krippendorff's alpha exists.

</details>